# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madihakomal75/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Feature Engineering & Pipeline Construction

We construct a clean feature vector for binary classification by processing numeric, temporal, and categorical fields:

1. **Temporal Ratios:** `staleness_ratio` (`days_since_last_update` / `content_age_days`) measures how much of a page's total lifespan has passed without an update.
2. **Engagement Ratios:** `clicks_per_impression` (CTR) and calculated `clicks_90d` evaluate user interaction efficiency.
3. **Categorical Encoding:** One-hot encoding is applied to `content_type` with `drop_first=True` to prevent multicollinearity.
4. **Missing Value Imputation:** Zero-fill imputation for numeric missing values and median filling for position data.

In [3]:
import os, sys, subprocess

REPO_URL = "https://github.com/madihakomal75/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# 1. Feature Engineering
df["staleness_ratio"] = (df["days_since_last_update"] / (df["content_age_days"] + 1)).clip(0, 1)
df["log_impressions"] = np.log1p(df["impressions_90d"].fillna(0))

# 2. Select raw feature subsets
numeric_features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count", "staleness_ratio", "log_impressions"]
categorical_features = ["content_type"] if "content_type" in df.columns else []

# 3. Categorical One-Hot Encoding
if categorical_features:
    df_encoded = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
else:
    df_encoded = df[numeric_features].copy()

# 4. Fill missing values
df_encoded = df_encoded.fillna(df_encoded.median(numeric_only=True))

print("Feature vector built successfully.")
print(f"Feature vector shape: {df_encoded.shape[0]:,} rows x {df_encoded.shape[1]} features")
df_encoded.head(3)

Feature vector built successfully.
Feature vector shape: 30,000 rows x 10 features


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,staleness_ratio,log_impressions,content_type_feedly article,content_type_keyword article
0,187,20,3803,10.6,0.76,3221.0,0.106383,8.243808,False,True
1,445,25,15320,20.3,0.05,2481.0,0.056054,9.636980,False,True
2,141,20,12581,36.5,0.09,3515.0,0.140845,9.440023,False,True


### Feature Availability & Processing Metadata

* **`content_age_days`**: Total days elapsed since publication. Fully available before prediction cutoff point.
* **`days_since_last_update`**: Days elapsed since the page was last updated. Available prior to prediction cutoff. Missing values are filled with `content_age_days`.
* **`staleness_ratio`**: Engineered feature computed as $\frac{\text{days\_since\_last\_update}}{\text{content\_age\_days} + 1}$. Quantifies relative freshness loss before prediction cutoff.
* **`impressions_90d` / `log_impressions`**: Aggregate impressions over the observation period prior to prediction time. Missing values zero-filled.
* **`avg_position`**: Average ranking position in search results during observation window. Missing values filled with median position.
* **`ctr`**: Average click-through rate observed in pre-prediction window. Missing values zero-filled.
* **`content_type`**: Categorical variable denoting page content format (e.g., blog post, landing page). One-hot encoded.

In [4]:
feature_audit = []
for col in df_encoded.columns:
    missing_cnt = df[col].isnull().sum() if col in df.columns else 0
    feature_audit.append({
        "Feature Name": col,
        "Type": df_encoded[col].dtype,
        "Missing Count": missing_cnt,
        "Missing %": f"{(missing_cnt / len(df)) * 100:.2f}%",
        "Available Before Prediction?": "Yes"
    })

audit_df = pd.DataFrame(feature_audit)
print("Feature Audit Table:")
audit_df

Feature Audit Table:


,Feature Name,Type,Missing Count,Missing %,Available Before Prediction?
0,content_age_days,int64,0,0.00%,Yes
1,days_since_last_update,int64,0,0.00%,Yes
2,impressions_90d,int64,0,0.00%,Yes
3,avg_position,float64,0,0.00%,Yes
4,ctr,float64,0,0.00%,Yes
5,word_count,float64,7699,25.66%,Yes
6,staleness_ratio,float64,0,0.00%,Yes
7,log_impressions,float64,0,0.00%,Yes
8,content_type_feedly article,bool,0,0.00%,Yes
9,content_type_keyword article,bool,0,0.00%,Yes


### Data & Target Leakage Audit

To ensure the model does not learn from unmeasured future outcomes or target definitions, we perform three checks:

1. **Target Correlation Check:** Identify features with direct mathematical relationships ($|r| > 0.90$) to `is_declining_label`.
2. **Post-Period Window Audit:** Confirm no features utilize metrics computed after the historical prediction cutoff date (`report_date`).
3. **Identifier Leakage Test:** Confirm non-predictive keys (`url_hash_id`, `client_hash_id`, `domain`) are excluded from feature matrices.

In [5]:
# Compute feature correlations with target label
correlations = df_encoded.apply(lambda col: col.corr(df["is_declining_label"])).abs()

print("Top 5 highest feature correlations with target (is_declining_label):")
print(correlations.sort_values(ascending=False).head(5).to_string())

# Flag potential leakage features (|r| > 0.85)
leakage_candidates = correlations[correlations > 0.85].index.tolist()
if leakage_candidates:
    print(f"\n WARNING: Potential leakage detected in features: {leakage_candidates}")
else:
    print("\n Clean: No features exhibit extreme target correlation (|r| > 0.85).")

Top 5 highest feature correlations with target (is_declining_label):
log_impressions                 0.177473
content_age_days                0.163882
staleness_ratio                 0.141349
content_type_feedly article     0.140455
content_type_keyword article    0.118346

 Clean: No features exhibit extreme target correlation (|r| > 0.85).


### Excluded Fields & Justification

1. **`trend_direction`**: Excluded because it forms the exact target proxy definition (`is_declining_label`). Retaining it would create absolute target leakage.
2. **`url_hash_id` / `client_hash_id`**: Excluded to prevent model overfitting to specific entity identifiers instead of learning generalizable feature patterns across domains.
3. **`search_volume`**: Excluded due to near-zero correlation ($r \approx 0.001$) with actual historical impressions, introducing uninformative noise.
4. **Post-cutoff performance metrics**: Excluded to enforce a strict temporal boundary and eliminate look-ahead bias.

In [6]:
excluded_summary = pd.DataFrame([
    {"Excluded Field": "trend_direction", "Reason": "Direct target definition source (causes 100% target leakage)"},
    {"Excluded Field": "url_hash_id", "Reason": "Unique identifier key (causes memorization & overfitting)"},
    {"Excluded Field": "client_hash_id", "Reason": "Grouping variable intended for holdout splits, not direct feature inputs"},
    {"Excluded Field": "search_volume", "Reason": "Near-zero correlation with actual organic impressions (noise reduction)"}
])

print("Summary of Excluded Fields:")
excluded_summary

Summary of Excluded Fields:


,Excluded Field,Reason
0,trend_direction,Direct target definition source (causes 100% t...
1,url_hash_id,Unique identifier key (causes memorization & o...
2,client_hash_id,"Grouping variable intended for holdout splits,..."
3,search_volume,Near-zero correlation with actual organic impr...
